In [17]:
using LinearAlgebra
using Distributions
using LaTeXStrings
using Printf
using FileIO
import JLD2
using DataFrames
using CSV

using Revise
using Newtrinos
using CairoMakie


In [18]:

osc_cfg = Newtrinos.osc.OscillationConfig(
    flavour=Newtrinos.osc.NND(),
    propagation=Newtrinos.osc.Basic(),
    states=Newtrinos.osc.All(),
    interaction=Newtrinos.osc.SI()
    )

osc = Newtrinos.osc.configure(osc_cfg)

physics = (; osc);


experiments = (

   katrin= Newtrinos.katrin.configure(physics),
);


par= Newtrinos.get_params(experiments)

[ Info: Loading Katrin data


(N = 20.0, m₀ = 0.01, r = 1.0, Δm²₂₁ = 7.53e-5, Δm²₃₁ = -0.0024, δCP = 1.0, θ₁₂ = 0.5872523687443223, θ₁₃ = 0.1454258194533693, θ₂₃ = 0.8556288707523761)

In [19]:

all_priors = Newtrinos.get_priors(experiments)

m0_values=[1e-1,1e-2,1e-3,1e-4]

N_treshold=[1,5, 50, 500]


for i in 1:length(m0_values)

    m0 =m0_values[i]
    par= merge(par, (m₀ =m0,))
    
    vars_to_scan = (r=31, N=31)  

    modified_priors = (
        N = Uniform((N_treshold[i]),(N_treshold[i]+200)) ,
        m₀ =all_priors.m₀,
        r = all_priors.r,
        
    
    

        Δm²₂₁ = par.Δm²₂₁,
        Δm²₃₁ = all_priors.Δm²₃₁,
        δCP = par.δCP,
        θ₁₂ = par.θ₁₂,
        θ₁₃ = all_priors.θ₁₃,
        θ₂₃ = par.θ₂₃
    )
        

    likelihood_NN = Newtrinos.generate_likelihood(experiments);

    result = Newtrinos.scan(likelihood_NN, modified_priors, vars_to_scan, par)


    JLD2.@save "/home/sofialon/Newtrinos.jl/plot_final/scans_new_scale_file/katrin_rN_NND_IO_m0=$m0.jld2" result
    

    img = CairoMakie.plot(result; title="Katrin - LogLikelihood r vs N, mo=$m0 NND IO", log=0, mass=0)
   
    save("/home/sofialon/Newtrinos.jl/plot_final/scans_new_scale/katrin_rN_NND_IO_m0=$m0.png", img)

end    


Progress: 100%|█████████████████████████████████████████| Time: 0:00:08
Progress: 100%|█████████████████████████████████████████| Time: 0:00:08
Progress: 100%|█████████████████████████████████████████| Time: 0:00:24
Progress: 100%|█████████████████████████████████████████| Time: 0:14:45
